# 5. Agents and tools

Plain RAG embeds the question once. That's weak for exact lookups (*"price of SKU FUR-SOF-NOR-LYSA-GRY"*) and
for questions that mix topics. An **agent** lets the model decide which tools to call, and in which order.

> **Course concepts used here**
>
| Course notebook | What we use it for |
|---|---|
| `L_LLM_APIs/5_tool_calling` | tool schemas, the tool-call → tool-result loop |
| `N_LangChain/05_intro-to-langchain-agents` | what an agent is |
| `N_LangChain/06_langchain-retrieval-agents` | retrieval as a tool |
| `N_LangChain/10_langchain-building-tools` | writing custom tools with `@tool` |

In [1]:
import os, sys, warnings
from pathlib import Path

# Run from the repo root so `ironstore_assistant` can be imported
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from ironstore_assistant import config
from ironstore_assistant.fake_llm import FakeLLM
from ironstore_assistant.rag import get_llm

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("LLM:", config.CHAT_MODEL if HAS_KEY else "scripted FakeLLM (no OPENAI_API_KEY found)")

LLM: scripted FakeLLM (no OPENAI_API_KEY found)


In [3]:
from ironstore_assistant.agent import make_tools
from ironstore_assistant.build_index import ensure_index
from ironstore_assistant.index import load_index
from ironstore_assistant.rag import Retriever, load_products

ensure_index()
retriever = Retriever(load_index(), load_products())
collected = []  # every document a tool returns ends up here
tools = make_tools(retriever, collected)

## 5.1 What the model sees

`@tool` turns a Python function into a JSON schema: the **name**, the **docstring** (the model reads it to decide
when to use the tool) and the **arguments**. Writing good docstrings *is* prompt engineering.

In [4]:
for t in tools:
    print(f"{t.name}{t.args}\n    {t.description.splitlines()[0]}\n")

search_policies{'query': {'title': 'Query', 'type': 'string'}, 'area': {'default': '', 'title': 'Area', 'type': 'string'}}
    Search IronStore internal policies and procedures (HR, Finance, IT, Legal, Sales,

search_products{'query': {'title': 'Query', 'type': 'string'}}
    Semantic search over the current product catalogs (descriptions, specs, categories).

lookup_product{'sku_or_name': {'title': 'Sku Or Name', 'type': 'string'}}
    Exact lookup of a product by SKU (e.g. FUR-SOF-NOR-LYSA-GRY) or by (part of) its

list_documents{'area': {'default': '', 'title': 'Area', 'type': 'string'}}
    List the available internal documents (title, ID, department, last review).



## 5.2 Calling the tools ourselves

An exact SKU lookup reads the product table parsed in notebook 1, then returns the matching catalog chunk (so the
answer can still cite a page).

In [5]:
lookup = {t.name: t for t in tools}["lookup_product"]
print(lookup.invoke({"sku_or_name": "FUR-SOF-NOR-LYSA-GRY"})[:400])

- IronStore Product Catalog — Furniture p.1: 1. Nordholm Lysa 3-Seater Sofa
SKU: FUR-SOF-NOR-LYSA-GRY
Category: Furniture > Living Room > Sofas
Short description: Three-seat fabric sofa with removable seat cushions and a solid timber frame.
Price: €899.00
Key specifications:
Brand: Nordholm
Upholstery: Polyester-linen blend, charcoal grey
Dimensions: 218 × 92 × 84 cm
Seat height: 46 cm
Seat depth:


The reused SKU from notebook 1 is handled explicitly:

In [6]:
print(lookup.invoke({"sku_or_name": "IS-BED-002"}))

- IronStore Product Catalog — Bedding & Linens p.2: IS-BED-002 — LumaCasa Washed Cotton Bedding Set, Sage, 200 × 200 cm
Category: Bedding & Linens > Duvet Cover Sets
Price: €69.90
Availability: In stock
Warehouse location: Lille, France — WH-FR02
Short description:
Softened cotton bedding set with a matte washed finish. Suitable for double beds using a 200 × 200 cm duvet.
Key specifications:
Set includes one duvet cover and two pillowcases
Duvet co
- IronStore Product Catalog — Home Essentials p.10: IS-BED-002 — Restwell Memory Foam Pillow
Category: Home Essentials > Bedroom > Pillows
Short description: Contoured memory-foam pillow with removable washable cover.
Price: €34.00
Availability: In stock
Warehouse location: DE-DUS
Key specifications:
Dimensions: 60 × 40 × 12 cm
Filling: Viscoelastic memory foam
Cover: Polyester and viscose blend
Cover closure: Zip
Cover washing temperature: Up to 4


In [7]:
print({t.name: t for t in tools}["list_documents"].invoke({"area": "Finance"}))

- Budget Approval Workflow (ID FIN-PR-014, Finance, last reviewed August 2026)
- IronStore Expense Reimbursement Policy (ID FIN-EXP-004, Finance, last reviewed August 2026)
- Payroll Processing Procedures (ID FIN-PAY-004, Finance, last reviewed August 2026)


## 5.3 The agent loop

Same loop as the tool-calling notebook, written by hand so every step is visible:

1. send the question + tool schemas to the model (`llm.bind_tools(tools)`),
2. if it answers with tool calls, run them and send the results back as `ToolMessage`s,
3. repeat until it says `DONE` (max 4 steps).

Everything the tools returned is then given to the **same** cited-answer step as notebook 3, so agent answers have
the same citations, confidence and validation.

In [8]:
from langchain_core.messages import AIMessage
from ironstore_assistant.agent import AgentAssistant
from ironstore_assistant.rag import Assistant, StructuredAnswer

question = "How much is the Nordholm Lysa 3-Seater Sofa, is it in stock, and can I expense furniture for my home office?"
if HAS_KEY:
    llm = get_llm()
else:  # script what a model typically does with this question: two tools, then DONE
    llm = FakeLLM(
        tool_script=[
            AIMessage("", tool_calls=[
                {"name": "lookup_product", "args": {"sku_or_name": "Nordholm Lysa"}, "id": "call_1"},
                {"name": "search_policies", "args": {"query": "home office furniture expense reimbursement"},
                 "id": "call_2"}]),
            AIMessage("DONE"),
        ],
        structured=StructuredAnswer(
            answer="The Nordholm Lysa 3-Seater Sofa costs €899.00 and is in stock (24 units, Rotterdam) [1]. "
                   "For the expense part, see the passages on home-office equipment [2].",
            citations=[1, 2], confidence="medium", answerable=True))

agent = AgentAssistant(Assistant(retriever.store, llm, retriever.products, multi_query=False))
resp = agent.ask(question)
print("tool calls:")
for step in agent.last_trace:
    print("  ", step)
print("\n" + resp.answer)
for s in resp.sources:
    print(f"  [{s.number}] {s.label()}")

tool calls:
   lookup_product({'sku_or_name': 'Nordholm Lysa'})
   search_policies({'query': 'home office furniture expense reimbursement'})

The Nordholm Lysa 3-Seater Sofa costs €899.00 and is in stock (24 units, Rotterdam) [1]. For the expense part, see the passages on home-office equipment [2].
  [1] IronStore Product Catalog — Furniture (p. 1)
  [2] IronStore Expense Reimbursement Policy (p. 4–5)


**RAG or agent?** RAG is one fast, predictable retrieval: good for most policy questions. The agent costs extra
LLM calls but wins on exact lookups and multi-part questions. The app lets you switch between the two.

Next: how do we know any of this works? → **notebook 6**.